<a href="https://colab.research.google.com/github/rizkiismail9a/data-science-2026-unsia/blob/main/Pertemuan_10_MuhamadRizkiIsmail_240401010126.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [14]:
# ------------
# Nama: Muhamad Rizki Ismail
# Kelas: IF401
# NIM: 240401010126
# ------------

# Ensamble Learning Hands-On Activity

In [15]:
# Muat dataset dari google drive

from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [16]:
import pandas as pd

# Tentukan path lengkap ke file CSV
file_path = '/content/drive/My Drive/Colab Notebooks/dataset-dummy/Telco-Customer-Churn.csv'

# Muat dataset ke dalam DataFrame
df = pd.read_csv(file_path)

# Tampilkan 5 baris pertama DataFrame
display(df.head())
print(df.shape)

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


(7043, 21)


In [23]:
# Info kolom-kolom
print(df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   object 
 1   gender            7043 non-null   object 
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   object 
 4   Dependents        7043 non-null   object 
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   object 
 7   MultipleLines     7043 non-null   object 
 8   InternetService   7043 non-null   object 
 9   OnlineSecurity    7043 non-null   object 
 10  OnlineBackup      7043 non-null   object 
 11  DeviceProtection  7043 non-null   object 
 12  TechSupport       7043 non-null   object 
 13  StreamingTV       7043 non-null   object 
 14  StreamingMovies   7043 non-null   object 
 15  Contract          7043 non-null   object 
 16  PaperlessBilling  7043 non-null   object 


In [20]:
# Cek apakah dataset bersifat imbalanced
print(df["Churn"].value_counts(normalize=True))

Churn
No     0.73463
Yes    0.26537
Name: proportion, dtype: float64


Berikut adalah fitur-fitur kategorikal yang dapat di-encoding dari dataset `df`:

*   `customerID`
*   `gender`
*   `Partner`
*   `Dependents`
*   `PhoneService`
*   `MultipleLines`
*   `InternetService`
*   `OnlineSecurity`
*   `OnlineBackup`
*   `DeviceProtection`
*   `TechSupport`
*   `StreamingTV`
*   `StreamingMovies`
*   `Contract`
*   `PaperlessBilling`
*   `PaymentMethod`
*   `TotalCharges` (perlu diperiksa lebih lanjut karena `df.info()` menunjukkannya sebagai object)
*   `Churn`

Catatan: Kolom `TotalCharges` saat ini bertipe `object`. Biasanya, ini menunjukkan adanya nilai non-numerik (seperti spasi kosong atau string) di dalamnya, dan perlu dikonversi ke tipe numerik setelah menangani nilai-nilai tersebut.

In [26]:
# Identifikasi fitur kategorikal untuk encoding (selain customerID dan TotalCharges)
categorical_features = [
    'gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines',
    'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
    'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract',
    'PaperlessBilling', 'PaymentMethod'
]

# --- Penanganan kolom TotalCharges ---
# Konversi 'TotalCharges' ke numerik, ganti nilai non-numerik dengan NaN
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

# Isi nilai NaN yang mungkin muncul (misalnya, dari string kosong) dengan 0
# Diasumsikan pelanggan baru belum memiliki 'TotalCharges'.
df['TotalCharges'].fillna(0, inplace=True)

# --- Lanjutkan dengan Encoding ---
# Lakukan One-Hot Encoding pada fitur kategorikal
df_encoded = pd.get_dummies(df, columns=categorical_features, drop_first=True)

# Encoding variabel target 'Churn' (Yes=1, No=0)
df_encoded['Churn'] = df_encoded['Churn'].map({'Yes': 1, 'No': 0})

# Hapus kolom 'customerID'
df_encoded = df_encoded.drop('customerID', axis=1)

# Tampilkan 5 baris pertama DataFrame setelah encoding
display(df_encoded.head())

# Tampilkan informasi DataFrame untuk melihat tipe data baru dan jumlah kolom
print(df_encoded.info())

/tmp/ipykernel_1085/1361296332.py:15: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['TotalCharges'].fillna(0, inplace=True)


,SeniorCitizen,tenure,MonthlyCharges,TotalCharges,Churn,gender_Male,Partner_Yes,Dependents_Yes,PhoneService_Yes,MultipleLines_No phone service,...,StreamingTV_No internet service,StreamingTV_Yes,StreamingMovies_No internet service,StreamingMovies_Yes,Contract_One year,Contract_Two year,PaperlessBilling_Yes,PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check
0,0,1,29.85,29.85,0,False,True,False,False,True,...,False,False,False,False,False,False,True,False,True,False
1,0,34,56.95,1889.50,0,True,False,False,True,False,...,False,False,False,False,True,False,False,False,False,True
2,0,2,53.85,108.15,1,True,False,False,True,False,...,False,False,False,False,False,False,True,False,False,True
3,0,45,42.30,1840.75,0,True,False,False,False,True,...,False,False,False,False,True,False,False,False,False,False
4,0,2,70.70,151.65,1,False,False,False,True,False,...,False,False,False,False,False,False,True,False,True,False


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 31 columns):
 #   Column                                 Non-Null Count  Dtype  
---  ------                                 --------------  -----  
 0   SeniorCitizen                          7043 non-null   int64  
 1   tenure                                 7043 non-null   int64  
 2   MonthlyCharges                         7043 non-null   float64
 3   TotalCharges                           7043 non-null   float64
 4   Churn                                  7043 non-null   int64  
 5   gender_Male                            7043 non-null   bool   
 6   Partner_Yes                            7043 non-null   bool   
 7   Dependents_Yes                         7043 non-null   bool   
 8   PhoneService_Yes                       7043 non-null   bool   
 9   MultipleLines_No phone service         7043 non-null   bool   
 10  MultipleLines_Yes                      7043 non-null   bool   
 11  Inte

In [28]:
from sklearn.model_selection import train_test_split

# Pisahkan X (kolom input) dan y(kolom target)
X = df_encoded.drop('Churn', axis=1)
y = df_encoded['Churn']

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)


In [29]:
from sklearn.ensemble import RandomForestClassifier

# Latih model
rf = RandomForestClassifier(n_estimators=300, class_weight='balanced', random_state=42)
rf.fit(X_tr, y_tr)

RandomForestClassifier(class_weight='balanced', n_estimators=300,
                       random_state=42)

In [30]:
from sklearn.metrics import classification_report, roc_auc_score

# Prediksi
prob = rf.predict_proba(X_te)[:, 1]
pred = (prob > 0.5).astype(int)

# Evaluasi
print(classification_report(y_te, pred))
print("ROC-AUC:", roc_auc_score(y_te, prob))

              precision    recall  f1-score   support

           0       0.83      0.90      0.86      1035
           1       0.63      0.49      0.56       374

    accuracy                           0.79      1409
   macro avg       0.73      0.70      0.71      1409
weighted avg       0.78      0.79      0.78      1409

ROC-AUC: 0.8259849130693121


Laporan klasifikasi ini memberikan metrik evaluasi yang detail untuk setiap kelas (0 untuk 'Tidak Churn' dan 1 untuk 'Churn'), serta agregasi keseluruhannya.

    precision:
        Untuk kelas 0 (Tidak Churn): 0.83 — Ini berarti dari semua pelanggan yang diprediksi model tidak akan churn, 83% di antaranya benar-benar tidak churn. Ini adalah tingkat ketepatan prediksi 'Tidak Churn'.
        Untuk kelas 1 (Churn): 0.63 — Ini berarti dari semua pelanggan yang diprediksi model akan churn, 63% di antaranya benar-benar churn. Ini adalah tingkat ketepatan prediksi 'Churn'.
    recall:
        Untuk kelas 0 (Tidak Churn): 0.90 — Ini berarti model berhasil mengidentifikasi 90% dari semua pelanggan yang sebenarnya tidak churn. Ini adalah tingkat keberhasilan model dalam menemukan semua kasus 'Tidak Churn'.
        Untuk kelas 1 (Churn): 0.49 — Ini berarti model hanya berhasil mengidentifikasi 49% dari semua pelanggan yang sebenarnya churn. Ini adalah tingkat keberhasilan model dalam menemukan semua kasus 'Churn'.
    f1-score:
        Metrik ini adalah rata-rata harmonik dari presisi dan recall. Ini memberikan keseimbangan antara presisi dan recall.
        Untuk kelas 0: 0.86 — Angka yang baik, menunjukkan model cukup seimbang dalam memprediksi 'Tidak Churn'.
        Untuk kelas 1: 0.56 — Angka ini relatif lebih rendah, menunjukkan ada ruang untuk perbaikan dalam memprediksi 'Churn', terutama karena recall-nya yang rendah.
    support:
        Untuk kelas 0: 1035 — Jumlah aktual pelanggan yang tidak churn dalam dataset pengujian.
        Untuk kelas 1: 374 — Jumlah aktual pelanggan yang churn dalam dataset pengujian.
    accuracy: 0.79 — Secara keseluruhan, 79% prediksi model adalah benr.
    macro avg:
        Presisi: 0.73, Recall: 0.70, F1-score: 0.71 — Ini adalah rata-rata presisi, recall, dan f1-score antar kelas tanpa mempertimbangkan jumlah sampel di setiap kelas. Ini memberikan gambaran yang lebih baik tentang kinerja model pada kedua kelas, terutama jika ada ketidakseimbangan kelas.
    weighted avg:
        Presisi: 0.78, Recall: 0.79, F1-score: 0.78 — Ini adalah rata-rata presisi, recall, dan f1-score yang memperhitungkan jumlah sampel di setiap kelas (bobot sesuai dengan 'support').

2. ROC-AUC: 0.8259849130693121

    ROC-AUC (Receiver Operating Characteristic - Area Under the Curve) adalah metrik yang mengukur kemampuan model untuk membedakan antara kelas positif dan negatif. Nilainya berkisar dari 0 hingga 1.
    0.826 adalah skor yang cukup baik. Ini menunjukkan bahwa model memiliki kemampuan yang baik untuk membedakan pelanggan yang churn dan tidak churn. Nilai 0.5 berarti model tidak lebih baik dari tebakan acak, sedangkan 1.0 berarti model sempurna.

Kesimpulan dari Evaluasi:

Model menunjukkan akurasi keseluruhan yang baik (79%) dan kemampuan diskriminatif yang kuat (ROC-AUC 0.826). Namun, ada kesenjangan dalam kemampuan model untuk memprediksi kasus 'Churn' (kelas 1). Meskipun presisi untuk kelas 1 lumayan (63%), recall-nya hanya 49%. Ini berarti model cenderung melewatkan banyak pelanggan yang sebenarnya akan churn. Ini adalah masalah umum pada dataset yang tidak seimbang, dan meskipun menggunakan class_weight='balanced', masih ada ruang untuk perbaikan dalam mengidentifikasi minoritas kelas 'Churn'.